<a href="https://colab.research.google.com/github/ProfessorDong/Deep-Learning-Course-Examples/blob/master/CNN_Examples/CNN_Demo2_ResNet_Skip_Connections.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Demo 2: ResNet — The Power of Skip Connections

**ELC5365 Deep Learning** | Baylor University

This demo demonstrates the **degradation problem** and how **residual connections** solve it.

We will:
1. Build a plain 20-layer CNN (no skip connections)
2. Build a ResNet-20 (with skip connections)
3. Train both on CIFAR-10 and compare results
4. Visualize gradient flow to understand WHY skip connections help
5. Explore the bottleneck design used in deeper ResNets

**Reference:** Kaiming He et al., "Deep Residual Learning for Image Recognition," *CVPR*, 2016.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---
## Section 1: The Degradation Problem

**Intuition:** A deeper network should be *at least* as good as a shallower one.
The extra layers could simply learn identity mappings. But in practice:

- A **plain 34-layer** network performs **worse** than a plain 18-layer network!
- This is NOT overfitting — the **training error** is also higher.
- The optimization itself fails for plain deep networks.

**ResNet's solution:** Make it easy to learn identity mappings by adding **skip connections**.
Instead of learning `H(x)` directly, learn the **residual** `F(x) = H(x) - x`, so:

$$y = F(x) + x$$

If the identity mapping is optimal, the network only needs to push `F(x)` toward zero.

---
## Section 2: Building a Plain Deep Network (No Skip Connections)

In [ ]:
class PlainBlock(nn.Module):
    """A basic block WITHOUT skip connection: conv -> BN -> ReLU -> conv -> BN -> ReLU"""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))  # No skip connection!
        return out


class PlainNet(nn.Module):
    """Plain deep network (no residual connections) for CIFAR-10."""
    def __init__(self, num_blocks=3):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        # 3 groups of blocks: 16 -> 32 -> 64 channels
        self.layer1 = self._make_layer(16, 16, num_blocks, stride=1)
        self.layer2 = self._make_layer(16, 32, num_blocks, stride=2)
        self.layer3 = self._make_layer(32, 64, num_blocks, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, 10)

    def _make_layer(self, in_ch, out_ch, num_blocks, stride):
        layers = [PlainBlock(in_ch, out_ch, stride)]
        for _ in range(1, num_blocks):
            layers.append(PlainBlock(out_ch, out_ch, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x).view(x.size(0), -1)
        return self.fc(x)

plain_net = PlainNet(num_blocks=3)
total_layers = 1 + 3*2*3 + 1  # conv1 + 3 groups * 2 conv/block * 3 blocks + fc
total_params = sum(p.numel() for p in plain_net.parameters())
print(f"PlainNet-20: {total_params:,} parameters, ~{total_layers} layers")

---
## Section 3: Building ResNet with Skip Connections

The key equation: **y = F(x) + x**

The only difference from PlainNet: we **add the input `x` to the output** of each block.

In [ ]:
class ResidualBlock(nn.Module):
    """Residual block WITH skip connection: output = F(x) + x"""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        # Projection shortcut when dimensions change
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels))

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)   # <-- THE SKIP CONNECTION!
        out = self.relu(out)
        return out


class ResNet(nn.Module):
    """ResNet for CIFAR-10 with skip connections."""
    def __init__(self, num_blocks=3):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(16, 16, num_blocks, stride=1)
        self.layer2 = self._make_layer(16, 32, num_blocks, stride=2)
        self.layer3 = self._make_layer(32, 64, num_blocks, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, 10)

    def _make_layer(self, in_ch, out_ch, num_blocks, stride):
        layers = [ResidualBlock(in_ch, out_ch, stride)]
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(out_ch, out_ch, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x).view(x.size(0), -1)
        return self.fc(x)

resnet = ResNet(num_blocks=3)
total_params_res = sum(p.numel() for p in resnet.parameters())
print(f"ResNet-20:   {total_params_res:,} parameters")
print(f"PlainNet-20: {total_params:,} parameters")
print(f"\nThe ONLY difference is the '+= self.shortcut(x)' line in the forward pass!")

---
## Section 4: Training Both Networks on CIFAR-10

Let's train both networks and watch the difference in real time.

In [ ]:
# Data loading with standard CIFAR-10 augmentation
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

print(f"Training samples: {len(trainset):,}")
print(f"Test samples:     {len(testset):,}")

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * inputs.size(0)
        correct += outputs.argmax(1).eq(targets).sum().item()
        total += inputs.size(0)
    return total_loss / total, 100. * correct / total

def test(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            correct += model(inputs).argmax(1).eq(targets).sum().item()
            total += inputs.size(0)
    return 100. * correct / total

print("Training and evaluation functions defined.")

In [ ]:
# Train both networks
NUM_EPOCHS = 50  # Use 160 for full training; 50 is enough to see the difference

results = {}
for name, model in [('PlainNet-20', PlainNet(3)), ('ResNet-20', ResNet(3))]:
    print(f"\n{'='*60}")
    print(f"Training {name}...")
    print(f"{'='*60}")
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[25, 38], gamma=0.1)

    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}
    start = time.time()

    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train_epoch(model, trainloader, optimizer, criterion)
        test_acc = test(model, testloader)
        scheduler.step()
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:3d}/{NUM_EPOCHS} | Loss: {train_loss:.4f} | "
                  f"Train Acc: {train_acc:.1f}% | Test Acc: {test_acc:.1f}%")

    elapsed = time.time() - start
    print(f"  Finished in {elapsed:.0f}s | Final Test Acc: {test_acc:.1f}%")
    results[name] = history

---
## Section 5: Results — Seeing the Degradation Problem

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Training Loss
ax1.plot(results['PlainNet-20']['train_loss'], 'b-', label='PlainNet-20', linewidth=2)
ax1.plot(results['ResNet-20']['train_loss'], 'r-', label='ResNet-20', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Training Loss', fontsize=12)
ax1.set_title('Training Loss Comparison', fontsize=14, fontweight='bold')
ax1.legend(fontsize=12)
ax1.grid(True, alpha=0.3)

# Test Accuracy
ax2.plot(results['PlainNet-20']['test_acc'], 'b-', label='PlainNet-20', linewidth=2)
ax2.plot(results['ResNet-20']['test_acc'], 'r-', label='ResNet-20', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Test Accuracy (%)', fontsize=12)
ax2.set_title('Test Accuracy Comparison', fontsize=14, fontweight='bold')
ax2.legend(fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

plain_best = max(results['PlainNet-20']['test_acc'])
res_best = max(results['ResNet-20']['test_acc'])
print(f"Best Test Accuracy:")
print(f"  PlainNet-20: {plain_best:.1f}%")
print(f"  ResNet-20:   {res_best:.1f}%")
print(f"  Improvement: +{res_best - plain_best:.1f}%")
print(f"\nResNet converges faster and achieves higher accuracy!")
print(f"Skip connections help gradients flow through the network.")

---
## Section 6: Visualizing Gradient Flow

Let's measure gradient magnitudes at each layer. In the plain network, gradients
**vanish** in earlier layers. ResNet maintains healthier gradient flow.

In [ ]:
def get_gradient_norms(model, loader, criterion):
    """Compute gradient norms for each convolutional layer."""
    model.train()
    inputs, targets = next(iter(loader))
    inputs, targets = inputs.to(device), targets.to(device)

    model.zero_grad()
    loss = criterion(model(inputs), targets)
    loss.backward()

    grad_norms = []
    names = []
    for name, param in model.named_parameters():
        if 'conv' in name and 'weight' in name and param.grad is not None:
            grad_norms.append(param.grad.norm().item())
            names.append(name.replace('.weight', ''))
    return names, grad_norms

# Compare gradient norms
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

for ax, (net_name, net_class) in zip([ax1, ax2], [('PlainNet-20', PlainNet), ('ResNet-20', ResNet)]):
    model = net_class(3).to(device)
    names, norms = get_gradient_norms(model, trainloader, nn.CrossEntropyLoss())
    colors_grad = plt.cm.RdYlGn(np.linspace(0, 1, len(norms)))
    ax.bar(range(len(norms)), norms, color=colors_grad)
    ax.set_xlabel('Layer (shallow -> deep)', fontsize=11)
    ax.set_ylabel('Gradient Norm', fontsize=11)
    ax.set_title(f'{net_name} Gradient Flow', fontsize=13, fontweight='bold')
    ax.set_xticks(range(0, len(norms), 3))

plt.tight_layout()
plt.show()

print("ResNet maintains more uniform gradients across all layers.")
print("The skip connections act as 'gradient highways' through the network.")

---
## Section 7: Going Deeper — Does ResNet Scale?

With skip connections, we can train much deeper networks successfully.
Let's compare ResNet-20 vs ResNet-56 (9 blocks per group instead of 3).

In [ ]:
# Compare different depths
depths = {
    'ResNet-20 (n=3)': ResNet(3),
    'ResNet-56 (n=9)': ResNet(9),
    'PlainNet-20 (n=3)': PlainNet(3),
    'PlainNet-56 (n=9)': PlainNet(9),
}

print(f"{'Model':<22} {'Params':>10} {'Layers':>8}")
print('-' * 42)
for name, model in depths.items():
    params = sum(p.numel() for p in model.parameters())
    n = int(name.split('n=')[1].split(')')[0])
    num_layers = 6 * n + 2
    print(f"{name:<22} {params:>10,} {num_layers:>8}")

print("\nWith skip connections, ResNet-56 trains well and achieves ~93% on CIFAR-10.")
print("Without them, PlainNet-56 degrades and performs WORSE than PlainNet-20!")
print("ResNets have been trained up to 1202 layers deep!")

---
## Section 8: The Bottleneck Design

For deeper ResNets (50, 101, 152), a **bottleneck** block is used:
- **1x1 conv** to reduce channels (e.g., 256 -> 64)
- **3x3 conv** at the reduced dimension
- **1x1 conv** to restore channels (64 -> 256)

This dramatically reduces computation while maintaining the same depth.

In [ ]:
class BasicBlock(nn.Module):
    """Standard block: [3x3, 64] x 2"""
    def __init__(self, channels=64):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)

    def forward(self, x):
        return self.conv2(self.conv1(x)) + x

class BottleneckBlock(nn.Module):
    """Bottleneck block: 1x1 (reduce) -> 3x3 -> 1x1 (expand)"""
    def __init__(self, channels=256, bottleneck=64):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, bottleneck, 1, bias=False)    # 256 -> 64
        self.conv2 = nn.Conv2d(bottleneck, bottleneck, 3, padding=1, bias=False)  # 64 -> 64
        self.conv3 = nn.Conv2d(bottleneck, channels, 1, bias=False)    # 64 -> 256

    def forward(self, x):
        return self.conv3(self.conv2(self.conv1(x))) + x

basic = BasicBlock(64)
bottleneck = BottleneckBlock(256, 64)

basic_params = sum(p.numel() for p in basic.parameters())
bottleneck_params = sum(p.numel() for p in bottleneck.parameters())

print("Parameter comparison (both process same spatial dimensions):")
print(f"  Basic Block     [3x3,64] x 2:              {basic_params:>8,} params")
print(f"  Bottleneck Block [1x1,64]->[3x3,64]->[1x1,256]: {bottleneck_params:>8,} params")
print(f"\n  Bottleneck has {bottleneck_params/basic_params:.1f}x {'more' if bottleneck_params > basic_params else 'fewer'} params,")
print(f"  but it operates on 256-channel feature maps (4x the representation power).")
print(f"\n  With bottleneck design:")
print(f"    34-layer ResNet -> 50-layer ResNet")
print(f"    ResNet-50, ResNet-101, ResNet-152 are all bottleneck designs.")

---
## Summary

| Concept | Key Insight |
|---------|-------------|
| **Degradation problem** | Plain deep nets perform worse (not overfitting — optimization fails) |
| **Skip connections** | y = F(x) + x makes identity mappings easy to learn |
| **Gradient flow** | Skip connections act as gradient highways |
| **Bottleneck design** | 1x1 convolutions reduce computation for deeper networks |

**ResNet was the first CNN to surpass human-level performance on ImageNet (3.57% error)!**